# 03 — Model Training

Trains the XGBoost pipeline: `SimpleImputer → StandardScaler → XGBClassifier`.

Expects the feature-engineered dataset to already exist at `data/processed/features.parquet`.
If not, run `02_features.ipynb` first (or the full pipeline from `01_data.ipynb`).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd

from conf.settings import PROCESSED_DIR, DATASET_FILE, TEST_SIZE, VAL_SIZE, RANDOM_STATE
from src.module.model.data import load_raw_data, preprocess
from src.module.model.features import build_features, get_feature_matrix
from src.module.model.model import CSWinnerModel
from utils.data_processing import split_data

## 1. Load Data

Load from the processed cache. If it doesn't exist, run preprocess + build_features first.

In [ ]:
cached = PROCESSED_DIR / DATASET_FILE
if cached.exists():
    df = pd.read_parquet(cached)
    print(f'Loaded from cache: {df.shape}')
else:
    raw = load_raw_data()
    df = preprocess(raw)

df = build_features(df)
print(f'After features: {df.shape}')

## 2. Feature Matrix & Split

Stratified split into train / validation / test sets.

In [ ]:
X, y = get_feature_matrix(df)

X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_STATE
)

print(f'Train:      {len(X_train):>6,} rows  |  class balance: {y_train.mean():.1%}')
print(f'Validation: {len(X_val):>6,} rows  |  class balance: {y_val.mean():.1%}')
print(f'Test:       {len(X_test):>6,} rows  |  class balance: {y_test.mean():.1%}')
print(f'Features:   {X_train.shape[1]}')

## 3. Cross-Validation (optional)

Stratified 5-fold CV on the training set. Each fold trains a separate pipeline and reports AUC.
Folds run sequentially — each XGBClassifier uses all CPU cores.

In [ ]:
model = CSWinnerModel()
cv_results = model.cross_validate(X_train, y_train)
print(f"\nCV AUC: {cv_results['mean_auc']:.4f} ± {cv_results['std_auc']:.4f}")

## 4. Train Final Model

Fit the full pipeline on the training set, evaluate on the validation set.

In [ ]:
model = CSWinnerModel()
model.fit(X_train, y_train, X_val=X_val, y_val=y_val)

## 5. Validation Metrics

In [ ]:
from src.module.model.evaluate import ModelEvaluator

evaluator = ModelEvaluator(model)
val_metrics = evaluator.compute_metrics(X_val, y_val)

print('Validation set:')
for k, v in val_metrics.items():
    print(f'  {k:12s}: {v:.4f}')

## 6. Save Model

In [ ]:
path = model.save()
print(f'Pipeline saved: {path}')

## 7. Train / Val / Test Summary

In [ ]:
train_metrics = evaluator.compute_metrics(X_train, y_train)
test_metrics  = evaluator.compute_metrics(X_test,  y_test)

summary = pd.DataFrame({
    'Train': train_metrics,
    'Val':   val_metrics,
    'Test':  test_metrics,
}).T

summary.style.format('{:.4f}').background_gradient(axis=0)